In [ ]:
!pip install transformers datasets torch scikit-learn

In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 1. DEFINE LOADING FUNCTION
# ==========================================
def load_data(filepath, label):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = [line.strip() for line in f.readlines() if line.strip()]
        return pd.DataFrame({'text': lines, 'label': label})
    except Exception as e:
        print(f"⚠️ Error loading {filepath}: {e}")
        return pd.DataFrame()

# ==========================================
# 2. LOAD TRAINING DATA (For Fine-Tuning)
# ==========================================
print("Loading Training Data...")

# Adjust these paths if your folder structure is different
base_train_path = "/kaggle/input/normalised-training-data/"

df_tr_joy = load_data(base_train_path + "normalized_td_joy.txt", 0)
df_tr_sad = load_data(base_train_path + "normalized_td_sadness.txt", 1)
df_tr_sarc = load_data(base_train_path + "normalized_td_sarcasm.txt", 2)
df_tr_ang = load_data(base_train_path + "training_data_anger.txt", 3)
df_tr_neu = load_data(base_train_path + "training_data_neutral.txt", 4)

# Combine Training Data
df_train_final = pd.concat([df_tr_joy, df_tr_sad, df_tr_sarc, df_tr_ang, df_tr_neu], axis=0)
# Shuffle training data (Crucial for Neural Networks)
df_train_final = df_train_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ Total Training Sentences: {len(df_train_final)}")

# ==========================================
# 3. LOAD TESTING DATA (For Evaluation)
# ==========================================
print("Loading Testing Data...")

base_test_path = "/kaggle/input/final-testing-data-n/"

df_te_joy = load_data(base_test_path + "testing_data_n_joy.txt", 0)
df_te_sad = load_data(base_test_path + "testing_data_n_sadnes.txt", 1)
df_te_sarc = load_data(base_test_path + "testing_data_n_sarcasm.txt", 2)
df_te_ang = load_data(base_test_path + "testing_data_n_angerr.txt", 3)
df_te_neu = load_data(base_test_path + "testing_data_n_neutral.txt", 4)

# Combine Testing Data
df_test_final = pd.concat([df_te_joy, df_te_sad, df_te_sarc, df_te_ang, df_te_neu], axis=0)

print(f"✅ Total Testing Sentences: {len(df_test_final)}")

# ==========================================
# 4. PREPARE LISTS FOR XLM-R
# ==========================================
# This is exactly what the Tokenizer needs:

train_texts = df_train_final['text'].astype(str).tolist()
train_labels = df_train_final['label'].tolist()

test_texts = df_test_final['text'].astype(str).tolist()
test_labels = df_test_final['label'].tolist()

print("\n🚀 Data is ready for Tokenization!")
print(f"Example Text: {train_texts[0]}")
print(f"Example Label: {train_labels[0]}")

In [ ]:
import torch
from transformers import XLMRobertaTokenizer
from torch.utils.data import Dataset

# 1. LOAD TOKENIZER
# This downloads the "Dictionary" that XLM-R uses to understand 100 languages.
tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

# 2. DEFINE THE DATASET CLASS
class RomanUrduDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        # ENCODING: Text -> Numbers
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True, # Adds [CLS] and [SEP] tokens
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',     # Return PyTorch Tensors (for GPU)
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 3. CREATE THE DATASET OBJECTS
print("Preparing Datasets...")
train_dataset = RomanUrduDataset(train_texts, train_labels, tokenizer)
test_dataset = RomanUrduDataset(test_texts, test_labels, tokenizer)
print(f"✅ Ready! Training samples: {len(train_dataset)}")

In [ ]:
from transformers import XLMRobertaForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

# 1. LOAD THE PRE-TRAINED MODEL
# num_labels=5 because you have Joy, Sad, Sarcasm, Anger, Neutral
print("Downloading Model...")
model = XLMRobertaForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=5)

# 2. DEFINE METRICS (How to measure success)
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1) # Convert probabilities to Class ID
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

# 3. TRAINING ARGUMENTS (The Settings)
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,              
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=64,
    fp16=True,                       # <--- TURBO MODE (Keep this True for T4 GPU)
    warmup_steps=500,                
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    
    # --- THE FIX IS HERE ---
    eval_strategy="epoch",           # Changed from 'evaluation_strategy' to 'eval_strategy'
    # -----------------------
    
    save_strategy="epoch",           
    load_best_model_at_end=True,     
    report_to="none"                 
)

# 4. THE TRAINER (The Coach)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,       
    compute_metrics=compute_metrics
)

# 5. START TRAINING
print("🚀 Starting Fine-Tuning... (This will take 15-20 mins)")
trainer.train()
print("Training Complete!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print("Running Final Evaluation...")

# 1. Predict
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

# 2. Text Report
label_names = ["Joy", "Sadness", "Sarcasm", "Anger", "Neutral"]
print("\n" + "="*50)
print("XLM-ROBERTA FINAL RESULTS")
print("="*50)
print(classification_report(test_labels, preds, target_names=label_names))

# 3. Confusion Matrix Visual
cm = confusion_matrix(test_labels, preds)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=label_names, yticklabels=label_names)
plt.title("XLM-RoBERTa Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.savefig("/kaggle/working/xlm_confusion_matrix.png") # Save it!
plt.show()

# 4. SAVE MODEL
model.save_pretrained("/kaggle/working/xlm_roberta_final")
tokenizer.save_pretrained("/kaggle/working/xlm_roberta_final")
print("✅ Model saved to /kaggle/working/xlm_roberta_final")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import classification_report

# ==========================================
# 1. GET PREDICTIONS & METRICS
# ==========================================
print("📊 Generating Metrics for XLM-RoBERTa...")

# Run prediction on the Test Set
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

# Define Names
label_names = ["Joy", "Sadness", "Sarcasm", "Anger", "Neutral"]

# Generate Report as a Dictionary (so we can grab numbers)
report = classification_report(test_labels, preds, target_names=label_names, output_dict=True)

# Extract F1 Scores and Counts (Support) automatically
f1_scores = [report[label]['f1-score'] for label in label_names]
counts = [report[label]['support'] for label in label_names]

# ==========================================
# 2. F1-SCORE BAR CHART
# ==========================================
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f1c40f', '#9b59b6'] # Blue, Red, Green, Yellow, Purple

plt.figure(figsize=(10, 6))
bars = plt.bar(label_names, f1_scores, color=colors, edgecolor='black', alpha=0.8)

# Add numbers on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.01, f"{yval:.2f}", 
             ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.title('XLM-RoBERTa Performance (F1-Score per Class)', fontsize=16)
plt.ylabel('F1 Score (Higher is Better)', fontsize=12)
plt.xlabel('Emotion Category', fontsize=12)
plt.ylim(0, 1.0) # F1 is always between 0 and 1
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Save
plt.savefig('/kaggle/working/xlm_f1_bar_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ XLM-R F1 Bar Chart saved!")

# ==========================================
# 3. DATASET DISTRIBUTION PIE CHART
# ==========================================
plt.figure(figsize=(8, 8))
plt.pie(counts, labels=label_names, autopct='%1.1f%%', startangle=140, 
        colors=['#ff9999','#66b3ff','#99ff99','#ffcc99', '#c2c2f0'],
        shadow=True, textprops={'fontsize': 12})

plt.title(f'Test Set Distribution (Total: {sum(counts)} Sentences)', fontsize=16)

# Save
plt.savefig('/kaggle/working/xlm_pie_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ XLM-R Pie Chart saved!")

In [ ]:
import shutil
import os

# 1. Define the folder you want to download
# (This matches the name you used in model.save_pretrained)
folder_name = "xlm_roberta_final" 

# 2. Check if it exists
if os.path.exists(f"/kaggle/working/{folder_name}"):
    print(f"✅ Found folder: {folder_name}")
    
    # 3. Zip it
    shutil.make_archive("my_emotion_model", 'zip', f"/kaggle/working/{folder_name}")
    print(f"🎉 Success! Created 'my_emotion_model.zip'. Download this file from the Output tab.")
else:
    print("❌ Folder not found. Did you run model.save_pretrained()?")